<a href="https://colab.research.google.com/github/palarunava/machine-learning-courses/blob/main/machine-learning-specialization/supervised-ml-regression-classification/C2_W4_Decision_Trees.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <u>Decision Trees</u>

## Classifiers
### How to split a node?
The objective is to maximize the purity of that node, i.e. have the samples classified more accurately than any other split.  
For classification problems, the decision of whether to split is taken by determining which split will cause the largest reduction in <u>Information Gain</u>. Information Gain can also be called reduction in impurity.  

$$
\text{Information Gain =}
H(p_1^{root}) - \left(
w^{left} H(p_1^{left}) +
w^{right} H(p_1^{right})
\right)
$$

$\text{where,}$  
$p_1^{left} = \text{fraction of examples in the left node that have a positive label}$  
$w^{left} = \text{fraction of examples that went to the left node from the parent node}$  
$p_1^{right} = \text{fraction of examples in the right node that have a positive label}$  
$w^{right} = \text{fraction of examples that went to the right node from the parent node}$  
$p_1^{root} = \text{fraction of examples that have a positive label at the parent/root node}$  

<i>Verdict</i>: Choose the split that results in the highest <u>Information Gain</u>.
<br><br>
### When do we stop splitting?
* When a node is 100% one class
* When splitting a node will result in the tree exceeding a maximum depth (Root node is Depth 0)
* Information gain from additional splits is less than threshold
* When number of examples in a node is below a threshold
<br>

### Measuring the purity of a node
Entropy is the measure of purity of a node.  
$p_0 = 1 - p_1$  
$Entropy, H(p_1) = -p_1log_2(p_1)-p_0log_2(p_0)$  
$\qquad \;\;\; or, H(p_1) = -p_1log_2(p_1)-(1-p_1)log_2(1-p_1)$  
<br>

### A few points to remember
* For features that are not binary and have categorical values the use of one-hot encoding is recommended.
* So a feature having three values say V1, V2, V3 would end up being three different features f_V1, f_V2 and f_V3 having True or False values.
* <b><u>One hot encoding</u></b> - If a categorical feature can take on k values, create k binary features (0 or 1 valued).

## Regressors
* For continuous valued features, use a threshold value that would give us the highest information gain. For e.g. differentiating between cats and dogs based on weight (which is a continuous valued feature) us a threshold value (say 6 kg, so splitting based on whether weight >= 6kg) that would maximize the information gain.
* For regression problems, the decision of whether to split is taken by determining which split will cause the largest reduction in Variance.

$$
\text{Reduction in Variance = }
Variance^{root}-(w^{left} Variance^{left} + w^{right} Variance^{right})
$$

In [46]:
import kagglehub
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from xgboost.callback import EarlyStopping

RANDOM_STATE = 55

In [29]:
# Download latest version
path = kagglehub.dataset_download("fedesoriano/heart-failure-prediction")

Using Colab cache for faster access to the 'heart-failure-prediction' dataset.


In [30]:
# df['ST_Slope'].unique()
df = pd.read_csv(path + '/heart.csv')
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [31]:
cat_variables = ['Sex',
  'ChestPainType',
  'RestingECG',
  'ExerciseAngina',
  'ST_Slope'
]

# This will replace the columns with the one-hot encoded ones and keep the columns outside 'columns' argument as it is.
df = pd.get_dummies(
    data = df,
    prefix = cat_variables,
    columns = cat_variables
)
df.head()

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease,Sex_F,Sex_M,ChestPainType_ASY,...,ChestPainType_NAP,ChestPainType_TA,RestingECG_LVH,RestingECG_Normal,RestingECG_ST,ExerciseAngina_N,ExerciseAngina_Y,ST_Slope_Down,ST_Slope_Flat,ST_Slope_Up
0,40,140,289,0,172,0.0,0,False,True,False,...,False,False,False,True,False,True,False,False,False,True
1,49,160,180,0,156,1.0,1,True,False,False,...,True,False,False,True,False,True,False,False,True,False
2,37,130,283,0,98,0.0,0,False,True,False,...,False,False,False,False,True,True,False,False,False,True
3,48,138,214,0,108,1.5,1,True,False,True,...,False,False,False,True,False,False,True,False,True,False
4,54,150,195,0,122,0.0,0,False,True,False,...,True,False,False,True,False,True,False,False,False,True


In [32]:
features = [x for x in df.columns if x not in 'HeartDisease'] ## Removing our target variable

X_train, X_val, y_train, y_val = train_test_split(df[features], df['HeartDisease'], train_size = 0.8, random_state = RANDOM_STATE)

# We will keep the shuffle = True since our dataset has not any time dependency.

print(f'train samples: {len(X_train)}')
print(f'validation samples: {len(X_val)}')
print(f'target proportion: {sum(y_train)/len(y_train):.4f}')

In [39]:
decision_tree_model = DecisionTreeClassifier(
    min_samples_split = 50,
    max_depth = 4,
    random_state = RANDOM_STATE
).fit(X_train,y_train)

print(f"Metrics train:\n\tAccuracy score: {accuracy_score(decision_tree_model.predict(X_train),y_train):.4f}")
print(f"Metrics validation:\n\tAccuracy score: {accuracy_score(decision_tree_model.predict(X_val),y_val):.4f}")

Metrics train:
	Accuracy score: 0.8665
Metrics validation:
	Accuracy score: 0.8696


# <u>Tree Ensembles</u>

## Random Forest

Given a training set of size m  
For b = 1 to B (where B is the number of decision trees that make up the random forest)
* Use sampling with replacement to create a new training set of size m
* At each node, when choosing a feature to use to split, if n features are available, pick a random subset of k (< n) features and allow the algorithm to choose from that subset of features. Typically, k = √n
* If n is the number of features, we will randomly select √n features
  of these features to train each individual tree.
* Train a decision tree on the new dataset

```python
RandomForestClassifier(
	n_estimators = 100, # B, number of trees in the forest
	max_depth = 16, # maximum depth of the tree
	min_samples_split = 10 # The minimum number of samples required to split an internal node.
	max_features = 15 # k (<n) ~ √n, The number of features to consider when looking for the best split when splitting a node
).fit(X_train,y_train)
```
<br>

## XGBoost - eXtreme Gradient Boosting

Given a training set of size m  
For b = 1 to B (where B is the number of decision trees that make up the random forest)  
* Use sampling with replacement to create a new training set of size m
* But instead of picking from all examples with equal (1/m) probability, make it more likely to pick examples that the previously trained trees misclassify
* Train a decision tree on the new dataset

```python
XGBClassifier(
	n_estimators = 500, # B, Number of trees in the classifier.
		# With 10 boosting rounds: The model builds 10 simple decision trees, where each tree learns from the mistakes the previous tree made.
		# Tree 1: Makes initial predictions (probably not great), Tree 2: Looks at what Tree 1 got wrong and tries to fix those errors, Tree 3: Looks at what Trees 1 and 2 got wrong and fixes those
		# By default, XGBClassifier uses 100 boosting rounds, meaning it trains 100 trees.
	n_jobs = 4, # Number of parallel threads used to run xgboost.
)
```

In [41]:
#@title Random Forest

random_forest_model = RandomForestClassifier(
    n_estimators = 100,
    max_depth = 16,
    min_samples_split = 10
).fit(X_train,y_train)

print(f"Metrics train:\n\tAccuracy score: {accuracy_score(random_forest_model.predict(X_train),y_train):.4f}\nMetrics test:\n\tAccuracy score: {accuracy_score(random_forest_model.predict(X_val),y_val):.4f}")

Metrics train:
	Accuracy score: 0.9319
Metrics test:
	Accuracy score: 0.8913


In [48]:
#@title XGBoost

xgb_model = XGBClassifier(n_estimators = 500, learning_rate = 0.1,verbosity = 1, random_state = RANDOM_STATE,
    early_stopping_rounds=10)
xgb_model.fit(
    X_train,
    y_train,
    eval_set = [(X_val, y_val)]
)

print(xgb_model.best_iteration)

print(f"Metrics train:\n\tAccuracy score: {accuracy_score(xgb_model.predict(X_train),y_train):.4f}\nMetrics test:\n\tAccuracy score: {accuracy_score(xgb_model.predict(X_val),y_val):.4f}")

[0]	validation_0-logloss:0.63094
[1]	validation_0-logloss:0.58776
[2]	validation_0-logloss:0.54737
[3]	validation_0-logloss:0.51575
[4]	validation_0-logloss:0.48697
[5]	validation_0-logloss:0.46492
[6]	validation_0-logloss:0.44364
[7]	validation_0-logloss:0.42846
[8]	validation_0-logloss:0.41043
[9]	validation_0-logloss:0.39866
[10]	validation_0-logloss:0.38696
[11]	validation_0-logloss:0.37605
[12]	validation_0-logloss:0.36824
[13]	validation_0-logloss:0.35850
[14]	validation_0-logloss:0.35219
[15]	validation_0-logloss:0.34592
[16]	validation_0-logloss:0.34132
[17]	validation_0-logloss:0.33401
[18]	validation_0-logloss:0.32793
[19]	validation_0-logloss:0.32269
[20]	validation_0-logloss:0.32071
[21]	validation_0-logloss:0.31819
[22]	validation_0-logloss:0.31809
[23]	validation_0-logloss:0.31676
[24]	validation_0-logloss:0.31413
[25]	validation_0-logloss:0.31394
[26]	validation_0-logloss:0.31254
[27]	validation_0-logloss:0.31146
[28]	validation_0-logloss:0.31246
[29]	validation_0-loglos